In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp


In [ ]:
spark = SparkSession.builder \
    .appName("StateOfData-Ingestao-Bronze") \
    .getOrCreate()


In [ ]:
BUCKET = "tech-challenge-state-of-data"

RAW_PATH = f"s3://{BUCKET}/raw"
BRONZE_PATH = f"s3://{BUCKET}/bronze/state_of_data"


In [ ]:
path_2022 = f"{RAW_PATH}/state_of_data_2022.csv"
path_2024 = f"{RAW_PATH}/state_of_data_2024.csv"
path_2025 = f"{RAW_PATH}/state_of_data_2025_2026.csv"


In [ ]:
df_2022 = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .csv(path_2022)

df_2024 = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .csv(path_2024)

df_2025 = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .csv(path_2025)


In [ ]:
df_2022 = df_2022 \
    .withColumn("ano_pesquisa", lit(2022)) \
    .withColumn("periodo_pesquisa", lit("2022")) \
    .withColumn("arquivo_origem", lit("state_of_data_2022.csv")) \
    .withColumn("data_ingestao", current_timestamp())

df_2024 = df_2024 \
    .withColumn("ano_pesquisa", lit(2024)) \
    .withColumn("periodo_pesquisa", lit("2024")) \
    .withColumn("arquivo_origem", lit("state_of_data_2024.csv")) \
    .withColumn("data_ingestao", current_timestamp())

df_2025 = df_2025 \
    .withColumn("ano_pesquisa", lit(2025)) \
    .withColumn("periodo_pesquisa", lit("2025-2026")) \
    .withColumn("arquivo_origem", lit("state_of_data_2025-2026.csv")) \
    .withColumn("data_ingestao", current_timestamp())


In [ ]:
print(f"2022: {df_2022.count()} registros")
print(f"2024: {df_2024.count()} registros")
print(f"2025-2026: {df_2025.count()} registros")


In [ ]:
df_2022.printSchema()


In [ ]:
df_2024.printSchema()


In [ ]:
df_2025.printSchema()


In [ ]:
df_2022.write \
    .mode("overwrite") \
    .parquet(f"{BRONZE_PATH}/2022")

df_2024.write \
    .mode("overwrite") \
    .parquet(f"{BRONZE_PATH}/2024")

df_2025.write \
    .mode("overwrite") \
    .parquet(f"{BRONZE_PATH}/2025_2026")


In [ ]:
bronze_2022 = spark.read.parquet(f"{BRONZE_PATH}/2022")
bronze_2024 = spark.read.parquet(f"{BRONZE_PATH}/2024")
bronze_2025 = spark.read.parquet(f"{BRONZE_PATH}/2025_2026")


In [ ]:
print(f"Bronze 2022: {bronze_2022.count()} registros")
print(f"Bronze 2024: {bronze_2024.count()} registros")
print(f"Bronze 2025-2026: {bronze_2025.count()} registros")


In [ ]:
bronze_2022.select(
    "ano_pesquisa",
    "periodo_pesquisa",
    "arquivo_origem",
    "data_ingestao"
).show(5, truncate=False)

bronze_2024.select(
    "ano_pesquisa",
    "periodo_pesquisa",
    "arquivo_origem",
    "data_ingestao"
).show(5, truncate=False)

bronze_2025.select(
    "ano_pesquisa",
    "periodo_pesquisa",
    "arquivo_origem",
    "data_ingestao"
).show(5, truncate=False)
